# Milestone 3 Lab, notebook 4 of 6: A to-do list and a helper, by hand

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the problems by hand | Recall the question, find the data problems yourself, and correct the answer key |
| 2. Checks in code | Write two quality checks as queries, and prove each one works |
| 3. Quarantine, and the figures before and after | Set the wrong records aside, recalculate, and confirm the totals add up |
| **4. A to-do list and a helper, by hand** (you are here) | Ask a model to explain the problems, then build an agent that organises its own work |
| 5. A harness works the queue | Give every problem to a ready-made agent harness, and score its reports |
| 6. The same question, with real warnings | Ask the analyst the leader's question, now that the warnings come from real checks |

Model requests in this notebook: **about 20**. A full run of this lab costs about 10 US cents.

## What this notebook is for

The checks found seven problems. Each one needs investigating: what caused it, and what should the data engineer do? That is slow, repetitive reading. It is a good job for AI.

AI's job here is narrow. It **explains**. It does not decide: the outcomes were applied by code in notebook 3, and nothing in this notebook can change them.

You will try two ways:

1. **The simple way:** paste every finding into one prompt and ask one question.
2. **An agent that organises its own work.** It keeps a to-do list and hands each finding to a fresh helper. You build both of those by hand, so you can see they are small.

In [1]:
import json
from pathlib import Path

import pandas as pd
from lab_helpers import OPENROUTER_BASE_URL, connect, save_result, show_doc
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

from retail_profit_stack.quality.desk import build_desk, score_reports
from retail_profit_stack.quality.investigate import HELPER_INSTRUCTIONS
from retail_profit_stack.transform.pipeline import run_pipeline

client, MODEL = connect()

Connected to OpenRouter. Default model: anthropic/claude-haiku-5.5


## Step 1: The design

Where AI is used in this milestone, and where it is not, from the **HLD**.

In [2]:
show_doc("HLD.md", "6. Where AI is used and where it is not")

> **From `docs/HLD.md`: 6. Where AI is used and where it is not**

| Task | Done by |
|---|---|
| Finding bad records | Code (the checks) |
| Deciding pass, quarantine, or needs review | Code, by a fixed rule per check |
| Setting records aside | Code |
| Calculating figures | Code (the transformations) |
| Reading a finding and its records, and naming the likely cause | AI agent |
| Planning the work and keeping track of it | AI agent, with a to-do list |
| Writing the report for the data engineer | AI agent |
| Checking the agent covered every finding, and scoring its causes | Code |
| Confirming a cause and fixing the data | A person |
| Answering the leader's question | The analyst agent, as before |

The investigator's reports are notes for a person. No code reads an outcome or a figure from them.

## Step 2: The desk

An investigator needs material to work from. `build_desk` makes a folder, the **desk**, with everything in it. First make sure the pipeline has run, so the findings exist.

In [3]:
run_pipeline("sql")

DESK = Path("desk_by_hand")
on_desk = build_desk(DESK)

print("Findings on the desk:", on_desk)
print("In the folder:", sorted(path.name for path in DESK.iterdir()))

Findings on the desk: ['QF-001', 'QF-002', 'QF-003', 'QF-004', 'QF-005', 'QF-006', 'QF-007']
In the folder: ['findings', 'reports', 'rulebook.md']


Three things are on the desk:

- `findings/` has one file per finding.
- `rulebook.md` is the data quality rulebook.
- `reports/` is empty. The investigator's reports go there.

Here is one finding file. It holds the finding, its bad records, and the related records a person would look up next, such as the order.

In [4]:
print((DESK / "findings" / "QF-006.json").read_text())

{
  "finding_id": "QF-006",
  "check_name": "event_out_of_order",
  "month": "2026-10",
  "product_id": "P003",
  "product_name": "Hydration pack",
  "outcome": "needs_review",
  "description": "Records for Hydration pack in 2026-10 dated before their order: 1. They are still counted, in the month of their own date.",
  "bad_records": [
    {
      "source_table": "refunds",
      "what_is_wrong": "Refund dated 2026-10-19, but its order is dated 2026-10-31.",
      "record": {
        "refund_id": "REF-0049",
        "order_item_id": "OI-02523",
        "product_id": "P003",
        "refund_date": "2026-10-19",
        "refund_amount": 81.0,
        "return_handling_cost": 8.0
      }
    }
  ],
  "related_records": {
    "orders": [
      {
        "order_id": "ORD-22432",
        "order_date": "2026-10-31",
        "status": "completed"
      }
    ],
    "all_lines_of_those_orders": [
      {
        "order_item_id": "OI-02522",
        "order_id": "ORD-22432",
        "product_id":

And here is part of the rulebook. For each check it lists the usual **causes**. Each cause has a **sign**: something visible in the records that points to it.

In [5]:
rulebook = (DESK / "rulebook.md").read_text()

print(rulebook[rulebook.index("## event_out_of_order"):rulebook.index("## refund_exceeds_line")])

## event_out_of_order

**What it means.** A refund or a shipment is dated before the order it belongs to. That cannot really happen.

**What the system has done.** Needs review. The record is still counted.

**Usual causes.**

| Cause | Sign |
|---|---|
| `order_recorded_late` | The refund or shipment is dated exactly one day before the order. The order was written up the day after it was placed. |
| `date_typed_wrong` | The refund or shipment is dated two or more days before the order. Someone typed a wrong date. |




Before reading on, decide for yourself. In the finding above, the refund is dated 2026-10-19 and its order 2026-10-31. Which cause is it?

Twelve days before the order is "two or more days", so the cause is `date_typed_wrong`.

We planted these problems, so we know the true cause of every finding. That is our answer key for this part of the lab.

## Step 3: The simple way, one model call

Put the rulebook and all seven finding files into one prompt. Build it in pieces.

In [6]:
finding_texts = []
for path in sorted((DESK / "findings").glob("*.json")):
    finding_texts.append(path.read_text())

question = (
    "You are a data engineer. Below are a data quality rulebook and seven findings. "
    "For each finding, choose the one cause from the rulebook whose sign matches the records. "
    "Use the cause names exactly as the rulebook writes them."
)

prompt = question + "\n\n# Rulebook\n\n" + rulebook + "\n\n# Findings\n\n" + "\n\n".join(finding_texts)

print(len(prompt), "characters")

20338 characters


We want the reply as data, so we ask for a structured reply, as in milestone 1: one cause for each finding.

In [7]:
class Cause(BaseModel):
    finding_id: str
    likely_cause: str
    evidence: str


class AllCauses(BaseModel):
    causes: list[Cause]

In [8]:
reply = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    response_format=AllCauses,
    max_tokens=4000,
    extra_body={"provider": {"require_parameters": True}},
)

one_call = reply.choices[0].message.parsed
print("Input tokens:", reply.usage.prompt_tokens, " Output tokens:", reply.usage.completion_tokens)

Input tokens: 8563  Output tokens: 857


> **If that cell reports an error**, run it again.

Put the reply in a table, then score it. `score_reports` adds the cause we planted and marks each row right or wrong.

In [9]:
one_call_table = pd.DataFrame([cause.model_dump() for cause in one_call.causes])
one_call_table["status"] = "report read"

one_call_scored = score_reports(one_call_table)
one_call_scored[["finding_id", "likely_cause", "planted_cause", "result"]]

,finding_id,likely_cause,planted_cause,result
0,QF-001,single_record_resent,single_record_resent,right
1,QF-002,single_record_resent,single_record_resent,right
2,QF-003,batch_loaded_twice,batch_loaded_twice,right
3,QF-004,monthly_cost_not_updated,monthly_cost_not_updated,right
4,QF-005,order_recorded_late,order_recorded_late,right
5,QF-006,date_typed_wrong,date_typed_wrong,right
6,QF-007,amount_typed_wrong,amount_typed_wrong,right


In [10]:
one_call_right = int((one_call_scored["result"] == "right").sum())

print(one_call_right, "of", len(one_call_scored), "right, with one model call")

7 of 7 right, with one model call


With a capable model and only seven findings, the simple way often gets every cause right. That is an honest result, and for seven findings it would do.

So why build anything more? Look at what one call cannot do as the work grows:

- **The prompt grows with the queue.** Seven findings took the input tokens you saw above. Seven hundred would not fit, and the model reads every finding while deciding each one.
- **It is all or nothing.** If the call fails or the reply is cut short, you have no results at all.
- **There is no trail.** You cannot see which finding the model was looking at when it went wrong.

An agent that works through the findings one at a time fixes all three. It needs three small things.

## Step 4: Files as a workspace

The first is the ability to read and write files on the desk. These are two ordinary Python functions. Each prints a line when it is called, so you can watch.

`desk_path` is a safety rule: a name that would lead outside the desk folder is refused. When something goes wrong, a tool returns a plain sentence saying so. The model reads that sentence and can try something else.

In [11]:
def desk_path(name: str) -> Path | None:
    path = (DESK / name.lstrip("/")).resolve()
    if DESK.resolve() not in path.parents:
        return None  # outside the desk
    return path


def read_file(name: str) -> str:
    """Read a file on the desk. name is a path such as findings/QF-001.json or rulebook.md."""
    print("      read_file", name)
    path = desk_path(name)
    if path is None or not path.exists():
        return "There is no such file on the desk."
    return path.read_text()


def write_file(name: str, content: str) -> str:
    """Write a file on the desk. name is a path such as reports/QF-001.json."""
    print("      write_file", name)
    path = desk_path(name)
    if path is None or not path.parent.exists():
        return "You cannot write there."
    path.write_text(content)
    return "Written."

Try them yourself, as the model would.

In [12]:
print(read_file("rulebook.md")[:200])

      read_file rulebook.md
# Data quality rulebook

Kestrel Ridge Outfitters. This rulebook is for whoever investigates a data quality finding.

A **finding** is a problem that a quality check found in the business's records. E


## Step 5: A to-do list

The second thing is a to-do list. It sounds grand. It is a Python list, and a function that replaces it.

Why does an agent need one? In a long job a model can lose track of what it has finished. Writing the plan down, and updating it, keeps it on course. It also lets **us** see the plan.

In [13]:
todo_list = {"to_do": [], "done": []}


def write_todos(to_do: list[str], done: list[str]) -> str:
    """Replace the to-do list. Give the items still to do, and the items that are done."""
    todo_list["to_do"] = to_do
    todo_list["done"] = done
    print("write_todos:  to do", to_do, " done", done)
    return "To-do list updated."

## Step 6: A helper

The third thing is a **helper**, also called a subagent. It is a tool that starts a **fresh agent** for one finding, lets it work, and passes back one line.

Why fresh? Each finding file is long. If one agent read all seven, the detail of the first would still be in its context while it worked on the seventh, costing tokens and attention on every step. A helper starts clean, reads one finding, writes one report, and is gone.

Here are the helper's instructions. They are the production ones.

In [14]:
print(HELPER_INSTRUCTIONS)

You are a data engineer at Kestrel Ridge Outfitters. Investigate the one finding you are given.

Read the finding's file in findings/. Read rulebook.md for the rules of its check.
Look at the records in the file and choose the one cause whose sign matches them.

Write your report to reports/<finding id>.json with exactly these fields:
- finding_id
- likely_cause: one cause name, exactly as the rulebook writes it
- evidence: one or two sentences saying what in the records points to that cause
- suggested_action: one sentence for the data engineer

Then reply with one line: the finding id and the cause.



To build an agent we need the model as a LangChain object, as in milestone 2.

In [15]:
import os

llm = ChatOpenAI(
    model=MODEL,
    base_url=OPENROUTER_BASE_URL,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    timeout=120,
    max_retries=2,
    max_tokens=6000,
)

Now the helper. Read it line by line: it builds a new agent with the two file tools, gives it one finding, and returns the last thing the agent said.

In [16]:
helper_model_calls = []


def investigate(finding_id: str) -> str:
    """Give one finding to a helper. It investigates the finding and writes its report file."""
    print("investigate", finding_id)
    helper = create_agent(model=llm, tools=[read_file, write_file], system_prompt=HELPER_INSTRUCTIONS)
    request = {"messages": [{"role": "user", "content": f"Investigate finding {finding_id}."}]}
    result = helper.invoke(request)
    replies = [message for message in result["messages"] if message.type == "ai"]
    helper_model_calls.append(len(replies))
    return replies[-1].text

Try the helper on its own, on one finding. Watch it read the finding, read the rulebook, and write a report.

In [17]:
print(investigate("QF-006"))

investigate QF-006
      read_file findings/QF-006.json
      read_file rulebook.md
      write_file reports/QF-006.json
QF-006: date_typed_wrong


> **If that cell reports an error**, run it again.

Here is the report it wrote.

In [18]:
print((DESK / "reports" / "QF-006.json").read_text())

{
  "finding_id": "QF-006",
  "likely_cause": "date_typed_wrong",
  "evidence": "Refund REF-0049 (order item OI-02523, Hydration pack) is dated 2026-10-19, but its order ORD-22432 is dated 2026-10-31, which is 12 days earlier than the order and not exactly one day before it. That gap of two or more days matches the date_typed_wrong sign, not order_recorded_late.",
  "suggested_action": "Correct the refund date on REF-0049 against the source refund documentation (the order date of 2026-10-31 is the likely reference point), then re-run the October 2026 figures for Hydration pack."
}



## Step 7: The agent that runs the queue

Now the main agent. It gets four tools: list the findings, the to-do list, the helper, and `write_file` for its closing note. It has **no tool to read a finding**. It can only delegate.

For this trial we rebuild the desk with only three findings. A cap like this keeps a trial run cheap.

In [19]:
on_desk = build_desk(DESK, max_findings=3)
helper_model_calls.clear()

print("Findings on the desk:", on_desk)

Findings on the desk: ['QF-001', 'QF-002', 'QF-003']


In [20]:
def list_findings() -> list[str]:
    """List the finding ids waiting on the desk."""
    return sorted(path.stem for path in (DESK / "findings").glob("*.json"))


MAIN_INSTRUCTIONS = """You are a data engineer working through data quality findings on your own.

1. Call list_findings to see what is waiting.
2. Call write_todos with one item for each finding.
3. For each finding, call investigate with its id. After each one, call write_todos again to move it to done.
4. When every finding is done, call write_file to write handover.md: a short note for the data engineer that groups findings with the same cause.
"""

main_agent = create_agent(
    model=llm,
    tools=[list_findings, write_todos, investigate, write_file],
    system_prompt=MAIN_INSTRUCTIONS,
)

Run it. In the output, lines that start at the left are the main agent. Indented lines are a helper reading and writing files.

In [21]:
request = {"messages": [{"role": "user", "content": "Please investigate the findings."}]}

result = main_agent.invoke(request, {"recursion_limit": 60})

main_model_calls = len([message for message in result["messages"] if message.type == "ai"])
print("\nMain agent model calls:", main_model_calls, " Helper model calls:", sum(helper_model_calls))

write_todos:  to do ['Investigate QF-001', 'Investigate QF-002', 'Investigate QF-003', 'Write handover.md']  done []
investigate QF-001
      read_file findings/QF-001.json
      read_file rulebook.md
      write_file      read_file reports/QF-001.json
 reports/QF-001.json
      read_file reports/QF-001.json
write_todos:  to do ['Investigate QF-002', 'Investigate QF-003', 'Write handover.md']  done ['Investigate QF-001']
investigate QF-002
      read_file findings/QF-002.json
      read_file rulebook.md
      write_file reports/QF-002.json
      write_file reports/QF-002.json.txt
write_todos:  to do ['Investigate QF-003', 'Write handover.md']  done ['Investigate QF-001', 'Investigate QF-002']
investigate QF-003
      read_file findings/QF-003.json
      read_file rulebook.md
      write_file reports/QF-003.json
write_todos:  to do ['Write handover.md', 'Manually delete stray file reports/QF-002.json.txt (created by helper, no delete tool)']  done ['Investigate QF-001', 'Investigate QF-

> **If that cell reports an error**, run it again.

Read the output from the top. You should see the plan being written, each finding handed to a helper, the helper's file reads and its report, and the to-do list changing as items move to done.

## Step 8: Score the reports

Each report is a file on the desk. Read them, in a plain loop.

In [22]:
rows = []
for finding_id in on_desk:
    report = json.loads((DESK / "reports" / f"{finding_id}.json").read_text())
    rows.append({"finding_id": finding_id, "likely_cause": report["likely_cause"], "evidence": report["evidence"], "status": "report read"})

by_hand_scored = score_reports(pd.DataFrame(rows))
pd.set_option("display.max_colwidth", 120)

by_hand_scored[["finding_id", "likely_cause", "planted_cause", "result", "evidence"]]

,finding_id,likely_cause,planted_cause,result,evidence
0,QF-001,single_record_resent,single_record_resent,right,"Only one record is duplicated: refund REF-0014 (order item OI-00514, order ORD-20499). The rulebook sign for single_..."
1,QF-002,single_record_resent,single_record_resent,right,"Three order lines (OI-01246, OI-01750, OI-01924) each appear twice across order_items, payment_fees and shipments, b..."
2,QF-003,batch_loaded_twice,batch_loaded_twice,right,"Twelve records are duplicated across four order items (OI-01433, OI-01434, OI-01436, OI-01437) in order_items, payme..."


And the note it left for the data engineer.

In [23]:
print((DESK / "handover.md").read_text())

# Handover: data quality findings QF-001 to QF-003

All three findings were investigated. The helper reports are brief and give only the finding type, so the root causes below are not confirmed.

## Grouping by likely cause

**Duplicate delivery from upstream (likely shared cause)**
- QF-001: single_record_resent. One record was sent more than once.
- QF-002: single_record_resent. Same pattern as QF-001.
- QF-003: batch_loaded_twice. A whole batch was loaded twice.

QF-001 and QF-002 share a type, which suggests a common resend source. QF-003 may be the same problem at batch level, for example a retried or replayed load. Check the upstream retry or replay behaviour and the load deduplication before treating these as separate issues.

## Cleanup needed

- The QF-002 helper created a stray file, `reports/QF-002.json.txt`, containing only the word "placeholder". It was not meant to be written. The desk has no delete tool, so please remove it by hand.

## Open items
- Confirm the root caus

In [24]:
by_hand_right = int((by_hand_scored["result"] == "right").sum())

save_result("by_hand", {
    "one_call": {"findings": len(one_call_scored), "right": one_call_right, "model_calls": 1},
    "by_hand": {"findings": len(by_hand_scored), "right": by_hand_right, "model_calls": main_model_calls + sum(helper_model_calls)},
})

Saved results/by_hand.json


## What we now know

1. One model call can explain a handful of findings. It does not scale, it is all or nothing, and it leaves no trail.
2. An agent that organises its own work needs three things: **files** to work in, a **to-do list**, and a **helper** with a clean start for each item.
3. Each of those is small. You wrote all three in a few lines.
4. The agent only wrote notes. It had no tool that could touch the data or an outcome.

**Next:** notebook 5. Software that gives an agent these three things ready-made is called a **harness**. You will use one on the whole queue.